# 第四阶段：STL（标准模板库）

## 实验 2：`std::string_view` —— 借用连续字符

实验 1 中的 `std::string` 拥有字符存储，而 `std::string_view` 只描述一段由其他对象拥有的连续字符。它通常可以理解为 pointer + length，不负责分配、复制或释放字符。

完成后你应该能够：

- 区分 `std::string` 的所有权与 `std::string_view` 的借用语义；
- 使用 `substr()`、`remove_prefix()` 和 `remove_suffix()` 创建零拷贝切片；
- 解释 `string_view` 为什么不保证以 NUL 结尾；
- 识别 owner 析构、重分配和临时对象造成的悬空 view；
- 在 C ABI 的 pointer + length 参数上建立受限生命周期的 view。

In [ ]:
// 本步骤：引入本实验需要的标准库和公开头文件。
#include <cassert>
#include <cstddef>
#include <iostream>
#include <string>
#include <string_view>

### 1. 同一个 view 可以读取不同来源

按值传递 `std::string_view` 很常见：复制的只是轻量视图，不会复制底层字符。函数只在调用期间读取参数，也不把 view 保存到调用结束之后。

In [ ]:
// 本步骤：通过代码演示“同一个 view 可以读取不同来源”并观察结果。
void print_view(std::string_view text)
{
    // operator<< 会按照 view 的 size() 输出，不要求末尾存在 NUL。
    std::cout
        << "text = " << text << '\n'
        << "size = " << text.size() << '\n';
}

{
    std::string owned = "Kotlin Native";

    // view 借用 owned 的字符；owned 必须在 print_view() 返回前保持有效。
    print_view(owned);

    // 字符串字面量具有静态生命周期，因此本次借用也是安全的。
    print_view("C++");

    std::string_view view = owned;
    assert(view == "Kotlin Native");
    assert(view.data() == owned.data());
}

`std::string` 与 `std::string_view` 的根本区别是所有权：

```text
std::string owner ── owns ──> character storage
                                  ▲
                                  │ borrows
                         std::string_view
                           pointer + length
```

`string_view` 析构时不会释放字符；复制 view 也不会复制字符。代价是它不能延长 owner 的生命周期，正确性取决于底层存储在整个借用期间始终有效。

### 2. `substr()` 创建零拷贝切片

`string_view::substr()` 返回另一个 view，只调整起始位置与长度，不创建新的 `std::string`。这很适合协议解析、命令行解析和日志字段切分。

In [ ]:
// 本步骤：通过代码演示“substr() 创建零拷贝切片”并观察结果。
{
    std::string owner = "Kotlin/Native";
    std::string_view whole = owner;

    // 下标 7 是 N 的位置；substr() 的结果仍然借用 owner。
    std::string_view native = whole.substr(7);

    print_view(native);

    assert(native == "Native");
    // 地址偏移证明切片指向原存储内部，而不是一份字符副本。
    assert(native.data() == owner.data() + 7);
}

`substr(pos, count)` 要求 `pos <= size()`，否则抛出 `std::out_of_range`；`count` 超过剩余长度时会自动截到末尾。切片不会拥有数据，所以它和原 view 一样受 owner 生命周期约束。

### 3. 调整 view 不会修改 owner

`remove_prefix(n)` 和 `remove_suffix(n)` 只改变 view 的边界。它们不会擦除 owner 中的字符，也不会分配内存。

In [ ]:
// 本步骤：通过代码演示“调整 view 不会修改 owner”并观察结果。
{
    std::string path = "/api/v1/users/";
    std::string_view trimmed = path;

    // 易错点：n 不能大于当前 size()，否则行为未定义。
    if (!trimmed.empty() && trimmed.front() == '/')
    {
        trimmed.remove_prefix(1);
    }

    if (!trimmed.empty() && trimmed.back() == '/')
    {
        trimmed.remove_suffix(1);
    }

    std::cout << "owner   = " << path << '\n';
    std::cout << "trimmed = " << trimmed << '\n';

    assert(path == "/api/v1/users/");
    assert(trimmed == "api/v1/users");
}

这里先用 `empty()` 检查，再访问 `front()` 或 `back()`，因为对空 view 调用二者是未定义行为。随后每次只移除一个已经确认存在的字符，因此满足 `n <= size()` 的前置条件。

这种接口把“解析游标”与原始数据分开：owner 保留完整输入，view 可以低成本地逐步缩小到当前字段。

### 4. `string_view` 不保证以 NUL 结尾

一个 view 可以只覆盖 owner 的中间部分。`data()` 返回起始指针，但 `data()[size()]` 不保证是 `\0`，甚至不保证该位置可读，因此不能把任意 `view.data()` 当作 C 字符串传给 `strlen()`、`printf("%s")` 等函数。

In [ ]:
// 本步骤：通过代码演示“stringview 不保证以 NUL 结尾”并观察结果。
{
    std::string owner = "Kotlin/Native";

    // 显式的 pointer + length 只借用前 6 个字符。
    std::string_view language(owner.data(), 6);

    std::cout << "view = " << language << '\n';
    assert(language == "Kotlin");

    // 这里只因我们知道 backing owner 的布局，才能安全查看下一字符。
    assert(owner[language.size()] == '/');

    // 错误：legacy_c_api(language.data()); 可能读取到 view 边界之外。
    // 需要 NUL 结尾时，显式创建一份拥有数据的 string。
    std::string nul_terminated(language);
    assert(nul_terminated.c_str()[nul_terminated.size()] == '\0');
}

构造 `nul_terminated` 会复制字符，但换来了独立所有权和末尾 NUL。是否值得复制取决于目标 API：

- 接受 pointer + length 的 API 可以直接使用 `view.data()` 和 `view.size()`；
- 只接受 NUL 结尾字符串的旧式 C API，需要确认 view 本来就覆盖完整 C 字符串，或先物化为 `std::string`；
- 即使 view 来自 `std::string`，只要 view 是其中一个切片，也不能推断 view 边界处存在 NUL。

### 5. 生命周期与失效规则

`string_view` 不会收到 owner 已销毁或已经重分配的通知。下面在 owner 扩容后立即重新绑定 view，并且绝不读取已经失效的旧借用。

In [ ]:
// 本步骤：通过代码演示“生命周期与失效规则”并观察结果。
{
    std::string owner = "Kotlin";
    std::string_view view = owner;

    // 请求超过旧 capacity 的容量会重分配，原 view 从此失效。
    owner.reserve(owner.capacity() + 100);

    // 关键步骤：先重新绑定，再读取；不能用旧 view 检查是否仍然可用。
    view = owner;
    assert(view == "Kotlin");

    // operator[] 修改已有元素不会改变存储位置，view 看到的不是快照。
    owner[0] = 'k';
    assert(view == "kotlin");
}

以下代码都能通过类型检查，却会产生悬空 view，因此只作为反例阅读：

```cpp
std::string_view bad_local()
{
    std::string local = "hello";
    return local; // 函数返回时 local 析构
}

std::string_view dangling = std::string("temporary");
// 本条语句结束时临时 string 析构，dangling 随即失效
```

安全的 view 必须满足：owner 比 view 活得久，并且借用期间没有让相关字符、指针或迭代器失效的操作。函数返回字符串字面量的 view 可以安全，因为字面量具有静态存储期；这不代表返回任意 view 都安全。

### 6. 在 C ABI 边界建立临时 view

C ABI 常用 pointer + length 表示输入缓冲区。C++ 实现可以在一次调用内部把它包装为 `string_view`，从而使用标准库的查找和切片接口，但不能让 view 逃逸到调用结束之后。

In [ ]:
// 本步骤：通过代码演示“在 C ABI 边界建立临时 view”并观察结果。
void sdk_inspect_text(const char* data, std::size_t size)
{
    // C ABI 契约必须保证 data 指向至少 size 个可读字符。
    assert(data != nullptr || size == 0);

    if (size == 0)
    {
        std::cout << "SDK input is empty\n";
        return;
    }

    // input 只在本函数内借用调用方缓冲区，不能保存到全局或异步任务。
    std::string_view input(data, size);
    print_view(input);
}

{
    std::string request = "run:model-v1";

    // request 覆盖整个调用，且调用期间不修改，因此借用有效。
    sdk_inspect_text(request.data(), request.size());
}

这段包装没有解决跨语言生命周期本身，只是把 C 契约映射成更易用的 C++ 只读视图：

```text
caller-owned buffer
        │ pointer + length
        ▼
C ABI call ──> temporary string_view ──> parse/read only
        │
        └─ return: view must not escape
```

如果 SDK 需要在返回后、其他线程或异步任务中继续使用输入，就必须复制到拥有数据的 `std::string`，或建立明确的外部缓冲区所有权协议。编码也需要单独约定；`string_view` 只知道 `char` 元素，不知道它们是否为 UTF-8。

### 本实验结论

`std::string_view` 是连续字符的非拥有视图，适合按值传参和零拷贝切片。它的优势来自“不管理资源”，最大的风险也来自“不管理资源”：owner 的析构、重分配或其他失效操作都可能留下悬空 view。

使用时持续检查三件事：

1. 谁拥有底层字符？
2. owner 是否覆盖 view 的完整使用周期，并在期间保持相关存储有效？
3. 下游 API 接受 pointer + length，还是错误地假设 `data()` 以 NUL 结尾？

只要 view 需要被保存、跨线程、跨异步边界或跨越 owner 的修改，就应重新评估是否应该复制为 `std::string`。